# Automatic Image Style Transfer — datasets 适配版

从 `gp_completed.ipynb` 复制，仅调整数据准备、划分、EDA 和预处理相关内容；模型结构、损失计算与训练流程保持原样；按目录要求更新结果路径及预训练权重位置。

**新数据目录**：`datasets/content_train`、`content_test`、`style_train/<artist_style>`、`style_test/<artist_style>`。文件夹名称是本地数据组织标签，不作为原始来源或授权的证明。

**划分规则**：保留给定测试集，从训练目录抽出 10% 作验证集；风格数据按艺术家—风格文件夹分层。完整解码检查后，按文件 SHA-256 去除完全相同的文件，测试集优先，避免相同文件跨划分泄漏。此检查不能排除不同压缩、裁剪版本的相似图。

**接口保持一致**：RGB、256×256、float32、[-1, 1]，保留原训练增强、Dataset 返回字段、batch size 和配对机制。训练输出放在 `DualEncoderStyleTransfer_result/`；预训练 VGG19 权重放在 `model/`。修改 `MODEL_NAME` 可以更换结果目录的模型名称前缀。

**运行**：按顺序运行。实验名称 `RUN_NAME` 与 `RUN_DIR` 在环境配置中统一定义；训练前的预览单元若找不到当前实验的 `best.pt`，会提示先训练并跳过预览。复制版本已清空旧执行输出，避免把旧数据结果当成新实验。后半部分保留的原说明（例如 213 张风格测试图、smoke run、128×128 等）属于旧实验记录；新数据计数与预处理尺寸以本 notebook 前半部分实际输出为准。没有执行新的完整训练，不能沿用旧训练结论。

**项目目录**：
```text
gp/
├── datasets/                         # 输入图片
│   ├── content_train/
│   ├── content_test/
│   ├── style_train/
│   └── style_test/
├── DualEncoderStyleTransfer_result/  # 本模型训练与测试结果
│   └── 1st_test/                     # RUN_NAME，每次实验可改名
│       ├── best.pt
│       ├── last.pt
│       └── test/                     # 测试指标与生成图片
├── model/
│   └── vgg19-dcbb9e9d.pth            # 冻结 VGG19 感知网络的预训练权重
└── gp_completed_new_dataset.ipynb
```
`model/` 中的权重若尚不存在，运行初始化单元时会优先复制已有 PyTorch 缓存，否则下载官方权重。结果与权重文件在实际运行相应单元后生成。


## 查看风格迁移结果 / Saved result preview

预览单元已移到第 12 节之前。先运行数据准备和模型定义，再运行预览；已有权重时无需重新训练。没有权重时会显示提示，可继续进行训练。


## 1. Environment Setup / 环境配置

This section checks the Python environment, project path, and available device.

本节检查 Python 环境、项目路径以及可用的训练设备。

In [ ]:
import sys
from pathlib import Path
from IPython.display import display
import torch
import random
import numpy as np
from PIL import Image, ImageOps
# No TensorFlow or torchvision is required.


In [ ]:
print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)

In [ ]:
# 自动寻找项目目录；移动项目后可直接修改下面候选路径。
_candidates = [Path.cwd(), *Path.cwd().parents,
               Path('/Users/haruka/Desktop/UNSW/9444/gp')]
PROJECT_ROOT = next((p for p in _candidates if (p / 'datasets').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('找不到 datasets，请将 notebook 放在 gp 目录下运行，或设置 PROJECT_ROOT。')
DATA_ROOT = PROJECT_ROOT / 'datasets'
MODEL_NAME = 'DualEncoderStyleTransfer'  # 自定义模型名称，决定结果文件夹前缀
RUN_ROOT = PROJECT_ROOT / f'{MODEL_NAME}_result'
PRETRAINED_ROOT = PROJECT_ROOT / 'model'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
PRETRAINED_ROOT.mkdir(parents=True, exist_ok=True)
# 训练、预览、测试共用同一个实验目录；切换实验只修改这里。
RUN_NAME = '1st_test'
RUN_DIR = RUN_ROOT / RUN_NAME
print('Current experiment:', RUN_DIR)
print('Project:', PROJECT_ROOT)
print('New dataset:', DATA_ROOT)
print('Experiment output:', RUN_ROOT)
print('Pretrained weights:', PRETRAINED_ROOT)


In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Selected device:", DEVICE)

## 2. Reproducibility / 实验可复现性

Random seeds are fixed so that dataset splitting, image pairing, and model
initialisation are as reproducible as possible.

固定随机种子，使数据划分、图像配对和模型参数初始化尽可能可复现。

In [ ]:
import random

import numpy as np


SEED = 42


def seed_everything(seed: int) -> None:
    """固定 Python、NumPy 和 PyTorch 的随机数生成器。"""

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)

print("Random seed:", SEED)

## 3. New Dataset Inventory / 新数据扫描与完整性检查

扫描实际图片，不依赖旧 `subset_manifest.csv`。完整解码、记录 EXIF 校正后的尺寸和原始颜色模式，再计算文件指纹。损坏图片不进入模型。

In [ ]:
DATA_ROOT = PROJECT_ROOT / 'datasets'
CONTENT_TRAIN_ROOT = DATA_ROOT / 'content_train'
CONTENT_TEST_ROOT = DATA_ROOT / 'content_test'
STYLE_TRAIN_ROOT = DATA_ROOT / 'style_train'
STYLE_TEST_ROOT = DATA_ROOT / 'style_test'
SOURCE_ROOTS = dict(content_train=CONTENT_TRAIN_ROOT, content_test=CONTENT_TEST_ROOT,
                    style_train=STYLE_TRAIN_ROOT, style_test=STYLE_TEST_ROOT)
for name, path in SOURCE_ROOTS.items():
    if not path.is_dir():
        raise FileNotFoundError(f'Missing {name}: {path}')
    print(name, path)

In [ ]:
import hashlib
import pandas as pd
from collections import defaultdict

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}
def list_images(root: Path):
    return sorted(p for p in root.rglob('*')
                  if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)

inventory, corrupt_images = [], []
for source_split, root in SOURCE_ROOTS.items():
    for path in list_images(root):
        try:
            with Image.open(path) as image:
                original_mode = image.mode
                corrected = ImageOps.exif_transpose(image)
                width, height = corrected.size
                corrected.convert('RGB').load()  # 完整解码，不能仅检查文件头
            inventory.append(dict(path=str(path), source_split=source_split,
                dataset='Content' if source_split.startswith('content') else 'Style',
                category='content' if source_split.startswith('content')
                         else path.relative_to(root).parent.as_posix(),
                original_width=width, original_height=height, mode=original_mode,
                sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
        except Exception as error:
            corrupt_images.append(dict(path=str(path), error=str(error)))
manifest = pd.DataFrame(inventory)
if manifest.empty:
    raise ValueError('新数据集没有可以解码的图片。')
manifest['aspect_ratio'] = manifest.original_width / manifest.original_height
print('Decoded images:', len(manifest), '| Corrupt images:', len(corrupt_images))
display(manifest.groupby(['source_split', 'category']).size().rename('images').to_frame())

In [ ]:
DATA_READY = all((manifest.source_split == name).any() for name in SOURCE_ROOTS)
if not DATA_READY:
    raise ValueError('四个输入目录都需要至少一张有效图片。')
print('Data ready:', DATA_READY)

## 4. Leakage-safe Splits / 去重与训练、验证、测试划分

给定的 `*_test` 作为测试集；从 `*_train` 的有效、去重图片中按固定种子抽取 10% 作验证。风格按艺术家—风格目录分层，至少两张图的组会保留训练图并提供验证图，单张组仅保留在训练集。测试图片不用于训练或选择超参数。

完全相同的文件以 SHA-256 判定，测试目录优先保留；同域内重复文件只使用一次。路径、数量、排除记录和指纹可通过 `manifest`、`duplicate_images` 检查；不移动或删除源图片。

In [ ]:
# 测试目录优先；只按内容/风格两域分别去重，不依据文件路径判定泄漏。
manifest['priority'] = manifest.source_split.str.endswith('_test').map({True: 0, False: 1})
manifest = manifest.sort_values(['priority', 'path']).reset_index(drop=True)
manifest['duplicate'] = manifest.duplicated(['dataset', 'sha256'], keep='first')
duplicate_images = manifest.loc[manifest.duplicate].copy()
manifest['split'] = 'excluded_duplicate'
manifest.loc[~manifest.duplicate, 'split'] = manifest.loc[~manifest.duplicate, 'source_split'].str.rsplit('_', n=1).str[-1]
print('Excluded exact duplicate files:', len(duplicate_images))
display(duplicate_images.groupby('source_split').size().rename('excluded').to_frame())

VAL_FRACTION = 0.10
def allocate_validation(indices, seed):
    indices = sorted(indices)
    random.Random(seed).shuffle(indices)
    count = min(len(indices)-1, max(1, round(len(indices)*VAL_FRACTION))) if len(indices) >= 2 else 0
    manifest.loc[indices[:count], 'split'] = 'val'

allocate_validation(manifest.index[(manifest.dataset == 'Content') & (manifest.split == 'train')], SEED)

In [ ]:
style_groups = {
    category: list(group.index)
    for category, group in manifest[(manifest.dataset == 'Style') & (manifest.split == 'train')].groupby('category')
}
for group_index, category in enumerate(sorted(style_groups)):
    allocate_validation(style_groups[category], SEED + group_index)
print('Style groups:', sorted(style_groups))

In [ ]:
def split_paths(dataset, split):
    return sorted(Path(p) for p in manifest.loc[(manifest.dataset == dataset) & (manifest.split == split), 'path'])
content_train, content_val, content_test = [split_paths('Content', s) for s in ('train', 'val', 'test')]
style_train, style_val, style_test = [split_paths('Style', s) for s in ('train', 'val', 'test')]
# 原始有效图片列表保留，便于完整性报告。
raw_content_train = [Path(p) for p in manifest.loc[manifest.source_split == 'content_train', 'path']]
raw_content_test = [Path(p) for p in manifest.loc[manifest.source_split == 'content_test', 'path']]
raw_style_images = [Path(p) for p in manifest.loc[manifest.dataset == 'Style', 'path']]
split_counts = pd.crosstab(manifest.loc[~manifest.duplicate, 'dataset'], manifest.loc[~manifest.duplicate, 'split'])
display(split_counts.reindex(columns=['train', 'val', 'test'], fill_value=0))

In [ ]:
for dataset in ('Content', 'Style'):
    subsets = [manifest[(manifest.dataset == dataset) & (manifest.split == s)] for s in ('train', 'val', 'test')]
    for subset in subsets:
        assert len(subset) > 0, f'{dataset}: empty split after cleaning'
    for i in range(3):
        for j in range(i+1, 3):
            assert set(subsets[i].path).isdisjoint(subsets[j].path)
            assert set(subsets[i].sha256).isdisjoint(subsets[j].sha256)
assert len(content_train) >= 4, '原 DataLoader 的 batch size=4 且 drop_last=True，需要至少四张内容训练图片。'
print('All six splits are non-empty; no shared paths or identical-file hashes within each domain.')

## 5. Exploratory Data Analysis / 新数据分析

分别报告原目录数量、清理后的实际划分、尺寸、长宽比、颜色模式及艺术家—风格分布。

In [ ]:
# category 表示原始艺术家—风格目录，例如 monet_impressionism，不能当作纯风格标签。

In [ ]:
import pandas as pd
from PIL import Image
from IPython.display import display

#打开第一张内容训练集图片
sample_content_path = content_train[0]

print("Image path:", sample_content_path)

with Image.open(sample_content_path) as image:
    print("Image size:", image.size)
    print("Image mode:", image.mode)

    content_preview = ImageOps.exif_transpose(image).convert("RGB")

display(content_preview)

In [ ]:
#打开第一张风格训练集图片
sample_style_path = style_train[0]

print("Style image path:", sample_style_path)

with Image.open(sample_style_path) as image:
    print("Style image size:", image.size)
    print("Style image mode:", image.mode)

    style_preview = ImageOps.exif_transpose(image).convert("RGB")

display(style_preview)

In [ ]:
# 清单来自当前扫描；无需 datasets 中预先提供 CSV。
display(manifest.head())
print('Inventory shape:', manifest.shape)
print('Columns:', manifest.columns.tolist())
display(manifest[['dataset', 'source_split', 'split', 'category', 'original_width', 'original_height']].head())

In [ ]:
print('Raw decoded counts:')
display(manifest.groupby('source_split').size().rename('images').to_frame())
print('Used after cleaning:')
display(manifest[~manifest.duplicate].groupby(['dataset', 'split']).size().rename('images').to_frame())

In [ ]:
display(manifest[['dataset', 'original_width', 'original_height', 'aspect_ratio', 'mode']].head())
display(pd.crosstab(manifest.source_split, manifest['mode']))

In [ ]:
# 兼容已有绘图变量名；这两张表是新数据的内容域与风格域。
coco_manifest = manifest[(manifest.dataset == 'Content') & ~manifest.duplicate].copy()
wikiart_manifest = manifest[(manifest.dataset == 'Style') & ~manifest.duplicate].copy()
print('Used content:', len(coco_manifest), '| Used style:', len(wikiart_manifest))

In [ ]:
dimension_columns = ['original_width', 'original_height', 'aspect_ratio']
for name, frame in [('Content', coco_manifest), ('Style', wikiart_manifest)]:
    print(name, 'image dimension statistics (EXIF corrected):')
    display(frame[dimension_columns].describe())

In [ ]:
style_counts = wikiart_manifest.category.value_counts()
print('Artist/style folder counts:')
display(style_counts.to_frame('images'))
display(pd.crosstab(wikiart_manifest.category, wikiart_manifest.split).reindex(columns=['train', 'val', 'test'], fill_value=0))
style_counts.plot.bar(figsize=(10, 4), title='Artist/style folder distribution (after deduplication)')
plt_module = __import__('matplotlib.pyplot', fromlist=['pyplot'])
plt_module.ylabel('Images')
plt_module.tight_layout()
plt_module.show()

In [ ]:
print('Missing inventory values:')
print(manifest.isna().sum())

In [ ]:
# 完整解码检查已在划分前完成，损坏文件已排除。
print('Corrupt/unreadable images excluded:', len(corrupt_images))
if corrupt_images:
    display(pd.DataFrame(corrupt_images))
print('Exact duplicate files excluded:', len(duplicate_images))

### 5.1 Data Visualisation / 数据可视化

Random image grids are used to inspect visual diversity and possible artefacts.
Histograms and bar charts are used to analyse aspect ratios and style imbalance.

随机图片网格用于检查图像多样性和潜在瑕疵。直方图和柱状图用于分析
长宽比分布以及风格类别不平衡。

In [ ]:
import math

import matplotlib.pyplot as plt
import seaborn as sns
#随机图片显示
def show_image_samples(image_paths:list[Path],
                       title:str,
                       number_of_images:int = 8,
                       seed: int = 42):
    if len(image_paths) == 0:
        print("No image available")
        return

    number_to_show = min(number_of_images, len(image_paths))

    sample_rng = random.Random(seed)
    selected_paths = sample_rng.sample(image_paths,number_to_show)
    number_of_columns = 4
    number_of_rows = math.ceil(number_to_show / number_of_columns)
    figure, axes = plt.subplots(
        number_of_rows,
        number_of_columns,
        figsize=(
            3 * number_of_columns,
            3 * number_of_rows,
        ),
    )
    axes = np.atleast_1d(axes).ravel()
    for axis, image_path in zip(
        axes,
        selected_paths,
    ):
        with Image.open(image_path) as image:
            rgb_image = ImageOps.exif_transpose(image).convert("RGB")

        axis.imshow(rgb_image)
        axis.set_title(
            image_path.parent.name,
            fontsize=9,
        )
        axis.axis("off")

    figure.suptitle(
        title,
        fontsize=14,
    )

    plt.tight_layout()
    plt.show()

show_image_samples(
    content_train,
    title="New dataset: content train",
    number_of_images=8,
    seed=SEED,
)

show_image_samples(
    style_train,
    title="New dataset: artist/style train",
    number_of_images=8,
    seed=SEED,
)

In [ ]:
#绘制长宽比分布
plt.figure(figsize=(10, 5))
sns.histplot(
    data=coco_manifest,
    x="aspect_ratio",
    bins=30,
    stat="density",
    alpha=0.5,
    label="Content",
)
sns.histplot(
    data=wikiart_manifest,
    x="aspect_ratio",
    bins=30,
    stat="density",
    alpha=0.5,
    label="Style",
)

# 保留全部长宽比范围，不隐藏极端图片。
plt.xlabel("Aspect Ratio")
plt.ylabel("Density")
plt.title("Image Aspect-Ratio Distribution")
plt.legend()
plt.show()

### EDA interpretation / 结论解释

实际数量见上方输出：原始扫描、损坏文件、重复排除、最终训练/验证/测试数量分别报告。训练与测试风格目录可能包含相同文件，因此需先按指纹去重，不能仅用不同路径判定无泄漏。

尺寸、长宽比与颜色模式不同，进入模型前统一 EXIF 方向与 RGB；继续使用原有随机裁剪/翻转训练增强和固定中心裁剪评估。中心裁剪可能丢失边缘内容，可通过下方预处理预览检查。

风格目录为艺术家与风格的组合标签；按这些标签分层验证。类别不平衡及同一艺术家在不同类别间出现，均需结合实际表格解释。本次检查只排除完全相同文件，不宣称排除了所有相似作品。

## 7. Image Preprocessing / 图像预处理

Before images can be passed into the neural network, they must be resized,
converted into PyTorch tensors, and normalised.

在将图片输入神经网络之前，我们需要统一图片尺寸，将其转换为
PyTorch 张量，并对像素值进行标准化。

Different transformations are used for training and evaluation:

- Training uses random augmentation to improve model generalisation.
- Validation and testing use deterministic preprocessing for fair comparison.

训练集使用随机数据增强，提高模型的泛化能力；验证集和测试集使用
固定的预处理方法，确保每次评估的结果具有可比性。

In [ ]:
# PIL → EXIF 校正 → RGB → 原有训练随机裁剪/翻转或评估中心裁剪
# → 256 × 256 → float32 Tensor [3, 256, 256] → [-1, 1]

In [ ]:
class ImageTransform:
    """PIL-only preprocessing; no torchvision or TensorFlow dependency."""
    def __init__(self, size=128, training=False):
        self.size, self.training = size, training

    def __call__(self, image):
        image = ImageOps.exif_transpose(image).convert('RGB')
        w, h = image.size
        if self.training:
            area = w * h
            for _ in range(10):
                target = random.uniform(.65, 1.) * area
                ratio = math.exp(random.uniform(math.log(.75), math.log(4/3)))
                cw, ch = round(math.sqrt(target * ratio)), round(math.sqrt(target / ratio))
                if 0 < cw <= w and 0 < ch <= h:
                    x, y = random.randint(0, w-cw), random.randint(0, h-ch)
                    image = image.crop((x, y, x+cw, y+ch))
                    break
            else:
                side = min(w, h)
                image = image.crop(((w-side)//2, (h-side)//2, (w+side)//2, (h+side)//2))
            image = image.resize((self.size, self.size), Image.Resampling.BILINEAR)
            if random.random() < .5:
                image = ImageOps.mirror(image)
        else:
            short = round(self.size * 1.12)
            scale = short / min(w, h)
            image = image.resize((round(w*scale), round(h*scale)), Image.Resampling.BILINEAR)
            w, h = image.size
            x, y = (w-self.size)//2, (h-self.size)//2
            image = image.crop((x, y, x+self.size, y+self.size))
        array = np.array(image, dtype=np.float32) / 127.5 - 1
        return torch.from_numpy(array.transpose(2, 0, 1).copy())


def build_transform(size=128, training=False):
    return ImageTransform(size, training)


IMAGE_SIZE = 256
train_transform = build_transform(IMAGE_SIZE, training=True)
eval_transform = build_transform(IMAGE_SIZE, training=False)


In [ ]:
# 打开一张训练内容图片
with Image.open(content_train[0]) as image:
    # 统一转换成 RGB，避免灰度图或带透明通道的图片造成错误
    sample_image = ImageOps.exif_transpose(image).convert("RGB")

    # 分别应用训练集和评估集预处理
    train_tensor = train_transform(sample_image)
    eval_tensor = eval_transform(sample_image)

# 查看处理后的 Tensor 信息
print("训练 Tensor 形状：", train_tensor.shape)
print("训练 Tensor 类型：", train_tensor.dtype)
print("训练 Tensor 最小值：", train_tensor.min().item())
print("训练 Tensor 最大值：", train_tensor.max().item())

print()

print("评估 Tensor 形状：", eval_tensor.shape)
print("评估 Tensor 类型：", eval_tensor.dtype)
print("评估 Tensor 最小值：", eval_tensor.min().item())
print("评估 Tensor 最大值：", eval_tensor.max().item())
assert train_tensor.shape == eval_tensor.shape == (3, IMAGE_SIZE, IMAGE_SIZE)
assert train_tensor.dtype == eval_tensor.dtype == torch.float32
assert torch.isfinite(train_tensor).all() and torch.isfinite(eval_tensor).all()
assert -1 <= train_tensor.min() <= train_tensor.max() <= 1
assert -1 <= eval_tensor.min() <= eval_tensor.max() <= 1
with Image.open(content_train[0]) as image:
    assert torch.equal(eval_transform(image), eval_transform(image))
print('Preprocessing shape/range/dtype and evaluation determinism verified.')

In [ ]:
#把 Tensor 转回可显示的图片
def denormalise_image(tensor):
    """
    将像素范围从 [-1, 1] 转回 [0, 1]，
    方便使用 Matplotlib 显示图片。

    参数：
        tensor：形状为 [C, H, W] 的图像 Tensor

    返回：
        反标准化后的 CPU Tensor
    """

    # detach()：切断梯度记录
    # cpu()：将 Tensor 移到 CPU
    # clamp()：限制数值范围，避免出现异常像素
    tensor = tensor.detach().cpu().clamp(-1.0, 1.0)

    # [-1, 1] → [0, 1]
    tensor = tensor * 0.5 + 0.5

    return tensor

# 创建一行三列的画布
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# 显示原始图片
axes[0].imshow(sample_image)
axes[0].set_title("Original")
axes[0].axis("off")

# 显示训练集随机预处理结果
axes[1].imshow(
    denormalise_image(train_tensor).permute(1, 2, 0)
)
axes[1].set_title("Training transform")
axes[1].axis("off")

# 显示固定的评估预处理结果
axes[2].imshow(
    denormalise_image(eval_tensor).permute(1, 2, 0)
)
axes[2].set_title("Evaluation transform")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 8. Dataset and DataLoader / 数据集与数据加载器

A custom PyTorch `Dataset` is used to load one content image and one style
image for every training sample.

我们使用自定义 PyTorch `Dataset`。每个训练样本包含一张内容图片和
一张风格图片。

During training, a random style image is selected for each content image.
This allows the model to observe different content-style combinations.

训练时，每张内容图片会随机搭配一张风格图片，使模型能够学习不同的
内容与风格组合。

The `DataLoader` groups multiple samples into a mini-batch and supplies them
to the neural network.

`DataLoader` 会把多个样本组合成一个小批次（mini-batch），并将其
提供给神经网络。

In [ ]:
from torch.utils.data import Dataset, DataLoader
class StyleTransferDataset(Dataset):
    def __init__(self, content_paths, style_paths, transform, random_pairing=True):
        self.content_paths, self.style_paths = list(content_paths), list(style_paths)
        if not self.content_paths or not self.style_paths:
            raise ValueError('Both content and style splits must contain images.')
        self.transform, self.training = transform, random_pairing
        # Fixed shuffled pairing covers every held-out style, rather than the first 100 sorted paths.
        self.eval_styles = self.style_paths.copy()
        random.Random(42).shuffle(self.eval_styles)

    def __len__(self):
        return len(self.content_paths) if self.training else max(len(self.content_paths), len(self.style_paths))

    def __getitem__(self, index):
        cp = self.content_paths[index % len(self.content_paths)]
        sp = random.choice(self.style_paths) if self.training else self.eval_styles[index % len(self.eval_styles)]
        with Image.open(cp) as image:
            content = self.transform(image)
        with Image.open(sp) as image:
            style = self.transform(image)
        return dict(content=content, style=style, content_path=str(cp), style_path=str(sp))



In [ ]:
train_dataset = StyleTransferDataset(
    content_paths=content_train,
    style_paths=style_train,
    transform=train_transform,
    random_pairing=True
)
val_dataset = StyleTransferDataset(
    content_paths=content_val,
    style_paths=style_val,
    transform=eval_transform,
    random_pairing=False
)
test_dataset = StyleTransferDataset(
    content_paths=content_test,
    style_paths=style_test,
    transform=eval_transform,
    random_pairing=False
)
# 打印三个数据集的大小
print("训练集样本数量：", len(train_dataset))
print("验证集样本数量：", len(val_dataset))
print("测试集样本数量：", len(test_dataset))

In [ ]:
 # 读取训练数据集中的第一个样本
sample = train_dataset[0]

print("返回的数据类型：", type(sample))
print("字典中的键：", sample.keys())

print()
print("内容图片 Tensor 形状：", sample["content"].shape)
print("风格图片 Tensor 形状：", sample["style"].shape)

print()
print("内容图片路径：", sample["content_path"])
print("风格图片路径：", sample["style_path"])

# 创建一行两列的画布
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

# 显示内容图片
axes[0].imshow(
    denormalise_image(sample["content"]).permute(1, 2, 0)
)
axes[0].set_title("Content image / 内容图片")
axes[0].axis("off")

# 显示风格图片
axes[1].imshow(
    denormalise_image(sample["style"]).permute(1, 2, 0)
)
axes[1].set_title("Style image / 风格图片")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch
# DataLoader: fixed held-out pairing and deterministic preprocessing
# 每个 batch 包含的样本数量
BATCH_SIZE = 4

# 为了初学和兼容 macOS，暂时使用主进程加载数据
NUM_WORKERS = 0

# 只有使用 NVIDIA CUDA 时才开启 pin_memory
USE_PIN_MEMORY = DEVICE.type == "cuda"

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=USE_PIN_MEMORY,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=USE_PIN_MEMORY,
    drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=USE_PIN_MEMORY,
    drop_last=False
)

print("训练 batch 数量：", len(train_loader))
print("验证 batch 数量：", len(val_loader))
print("测试 batch 数量：", len(test_loader))

In [ ]:
# iter() 将 DataLoader 转换为迭代器
train_iterator = iter(train_loader)

# next() 从迭代器中取出第一个 batch
batch = next(train_iterator)

print("batch 中的键：", batch.keys())

print()
print("内容图片 batch 形状：", batch["content"].shape)
print("风格图片 batch 形状：", batch["style"].shape)

print()
print("第一张内容图片路径：")
print(batch["content_path"][0])

print()
print("第一张风格图片路径：")
print(batch["style_path"][0])

In [ ]:
# 最多显示 4 组内容—风格图片
number_to_show = min(4, batch["content"].shape[0])

# 每一行是一组内容图片和风格图片
fig, axes = plt.subplots(
    number_to_show,
    2,
    figsize=(7, 3 * number_to_show),
)

for row in range(number_to_show):
    # 显示内容图片
    content_image = denormalise_image(
        batch["content"][row]
    ).permute(1, 2, 0)

    axes[row, 0].imshow(content_image)
    axes[row, 0].set_title(f"Content {row + 1} / 内容图片")
    axes[row, 0].axis("off")

    # 显示风格图片
    style_image = denormalise_image(
        batch["style"][row]
    ).permute(1, 2, 0)

    axes[row, 1].imshow(style_image)
    axes[row, 1].set_title(f"Style {row + 1} / 风格图片")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()

## 9. Generator Building Blocks / 生成器基础模块

The generator converts a content image and a style image into a stylised image.

生成器接收一张内容图片和一张风格图片，并输出风格迁移后的图片。

Before implementing the complete generator, we first define reusable neural
network components:

1. Convolution block
2. Residual block

在实现完整生成器之前，先编写可以重复使用的神经网络组件：

1. 卷积模块
2. 残差模块

In [ ]:
# nn 包含神经网络中常用的层和模块
import torch.nn as nn

# F 包含一些不需要保存参数的神经网络函数
import torch.nn.functional as F
class ConvBlock(nn.Module):
    """
    可重复使用的卷积模块。

    处理顺序：
        Reflection Padding
        → Convolution
        → Instance Normalisation
        → Activation
    """

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
        stride=1,
        padding=1,
        use_normalisation=True,
        use_activation=True,
    ):
        """
        参数：
            in_channels：
                输入特征图的通道数量。

            out_channels：
                输出特征图的通道数量。

            kernel_size：
                卷积核尺寸，默认是 3 × 3。

            stride：
                卷积移动步长。
                stride=1 通常保持图片尺寸。
                stride=2 通常把高度和宽度缩小一半。

            padding：
                在图片边缘增加的像素数量。

            use_normalisation：
                是否使用 Instance Normalisation。

            use_activation：
                是否使用 ReLU 激活函数。
        """

        # 必须首先初始化父类 nn.Module
        super().__init__()

        # 使用反射填充减少图片边缘的人工痕迹
        self.padding = nn.ReflectionPad2d(padding)

        # 创建二维卷积层
        self.convolution = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=0,
        )

        # 根据参数决定是否使用 Instance Normalisation
        if use_normalisation:
            self.normalisation = nn.InstanceNorm2d(
                num_features=out_channels,
                affine=True,
            )
        else:
            # Identity 不改变输入
            self.normalisation = nn.Identity()

        # 根据参数决定是否使用 ReLU
        if use_activation:
            self.activation = nn.ReLU(inplace=True)
        else:
            self.activation = nn.Identity()

    def forward(self, x):
        """
        定义数据通过该模块的顺序。
        """

        x = self.padding(x)
        x = self.convolution(x)
        x = self.normalisation(x)
        x = self.activation(x)

        return x

In [ ]:
# 创建一个测试卷积模块
test_conv_block = ConvBlock(
    in_channels=3,
    out_channels=64,
    kernel_size=3,
    stride=1,
    padding=1,
)

# 创建两张随机 RGB 图片
# 形状：[batch, channel, height, width]
test_input = torch.randn(2, 3, 128, 128)

# 将图片输入卷积模块
test_output = test_conv_block(test_input)

print("输入形状：", test_input.shape)
print("输出形状：", test_output.shape)

In [ ]:
# stride=2 的卷积模块
test_downsample_block = ConvBlock(
    in_channels=64,
    out_channels=128,
    kernel_size=3,
    stride=2,
    padding=1,
)

# 将上一个模块的输出传入下采样模块
downsampled_output = test_downsample_block(test_output)

print("下采样前：", test_output.shape)
print("下采样后：", downsampled_output.shape)

In [ ]:
class ResidualBlock(nn.Module):
    """
    残差模块。

    输入和输出的形状保持不变：
        output = input + learned_features
    """

    def __init__(self, channels):
        """
        参数：
            channels：
                输入和输出的通道数量。
        """

        super().__init__()

        # 第一层：卷积 + 标准化 + ReLU
        self.conv1 = ConvBlock(
            in_channels=channels,
            out_channels=channels,
            kernel_size=3,
            stride=1,
            padding=1,
            use_normalisation=True,
            use_activation=True,
        )

        # 第二层不使用激活函数
        # 因为后面还要与原始输入相加
        self.conv2 = ConvBlock(
            in_channels=channels,
            out_channels=channels,
            kernel_size=3,
            stride=1,
            padding=1,
            use_normalisation=True,
            use_activation=False,
        )

        # 相加后使用 ReLU
        self.activation = nn.ReLU(inplace=True)

    def forward(self, x):
        """
        数据通过残差模块的过程。
        """

        # 保存原始输入，也叫 shortcut 或 skip connection
        residual = x

        # 学习需要修改的特征
        output = self.conv1(x)
        output = self.conv2(output)

        # 将学习到的特征与原始输入相加
        output = output + residual

        # 应用激活函数
        output = self.activation(output)

        return output

In [ ]:
# 创建一个具有 128 个通道的残差模块
test_residual_block = ResidualBlock(channels=128)

# 创建测试输入
test_features = torch.randn(2, 128, 64, 64)

# 输入残差模块
residual_output = test_residual_block(test_features)

print("残差模块输入形状：", test_features.shape)
print("残差模块输出形状：", residual_output.shape)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

test_conv_block = test_conv_block.to(device)
test_residual_block = test_residual_block.to(device)

print("卷积模块所在设备：")
print(next(test_conv_block.parameters()).device)

print()

print("残差模块所在设备：")
print(next(test_residual_block.parameters()).device)

In [ ]:
# 从 DataLoader 读取一个 batch
batch = next(iter(train_loader))

# 取出内容图片，并移动到训练设备
content_images = batch["content"].to(device)

# 创建一个新的卷积模块并移动到相同设备
input_block = ConvBlock(
    in_channels=3,
    out_channels=64,
    kernel_size=7,
    stride=1,
    padding=3,
).to(device)

# 将真实内容图片输入卷积模块
content_features = input_block(content_images)

print("输入图片形状：", content_images.shape)
print("卷积特征形状：", content_features.shape)
print("图片所在设备：", content_images.device)
print("模型特征所在设备：", content_features.device)

## 10. Content and Style Encoders / 内容与风格编码器

The encoders convert RGB images into compact feature representations.

编码器负责把 RGB 图片转换成更加紧凑的特征表示。

The content encoder focuses on spatial structure and semantic information,
while the style encoder learns colours, textures, brush strokes, and other
artistic characteristics.

内容编码器主要提取空间结构和语义信息；风格编码器主要提取颜色、
纹理、笔触以及其他艺术特征。

Both encoders have the same architecture but use independent learnable
parameters.

两个编码器使用相同的网络结构，但拥有相互独立的可学习参数。

In [ ]:
class ImageEncoder(nn.Module):
    """
    通用图像编码器。

    输入形状：
        [B, 3, 128, 128]

    输出形状：
        [B, 256, 32, 32]
    """

    def __init__(
        self,
        base_channels=64,
        number_of_residual_blocks=2,
    ):
        """
        参数：
            base_channels：
                第一层卷积输出的通道数量。

            number_of_residual_blocks：
                编码器末尾使用的残差模块数量。
        """

        super().__init__()

        # 第一层使用较大的 7 × 7 卷积核
        # 用于提取原始图片的浅层特征
        self.input_block = ConvBlock(
            in_channels=3,
            out_channels=base_channels,
            kernel_size=7,
            stride=1,
            padding=3,
            use_normalisation=True,
            use_activation=True,
        )

        # 第一次下采样
        # [B, 64, 128, 128] → [B, 128, 64, 64]
        self.downsample1 = ConvBlock(
            in_channels=base_channels,
            out_channels=base_channels * 2,
            kernel_size=3,
            stride=2,
            padding=1,
            use_normalisation=True,
            use_activation=True,
        )

        # 第二次下采样
        # [B, 128, 64, 64] → [B, 256, 32, 32]
        self.downsample2 = ConvBlock(
            in_channels=base_channels * 2,
            out_channels=base_channels * 4,
            kernel_size=3,
            stride=2,
            padding=1,
            use_normalisation=True,
            use_activation=True,
        )

        # 创建多个残差模块
        self.residual_blocks = nn.Sequential(
            *[
                ResidualBlock(base_channels * 4)
                for _ in range(number_of_residual_blocks)
            ]
        )

    def forward(self, x):
        """
        定义图片经过编码器的顺序。
        """

        x = self.input_block(x)
        x = self.downsample1(x)
        x = self.downsample2(x)
        x = self.residual_blocks(x)

        return x

In [ ]:
# 内容编码器：提取内容图片的结构和语义特征
content_encoder = ImageEncoder(
    base_channels=64,
    number_of_residual_blocks=2,
).to(device)

# 风格编码器：提取风格图片的颜色和纹理特征
style_encoder = ImageEncoder(
    base_channels=64,
    number_of_residual_blocks=2,
).to(device)

print("内容编码器创建完成。")
print("风格编码器创建完成。")

In [ ]:
# 从训练 DataLoader 中读取一个 batch
batch = next(iter(train_loader))

# 取出内容图片和风格图片
content_images = batch["content"].to(device)
style_images = batch["style"].to(device)

# 当前只是测试前向传播，所以关闭梯度记录
with torch.no_grad():
    content_features = content_encoder(content_images)
    style_features = style_encoder(style_images)

print("内容图片形状：", content_images.shape)
print("内容特征形状：", content_features.shape)

print()

print("风格图片形状：", style_images.shape)
print("风格特征形状：", style_features.shape)

In [ ]:
# 只取一张图片，方便观察形状
single_image = content_images[:1]

with torch.no_grad():
    feature_1 = content_encoder.input_block(single_image)
    feature_2 = content_encoder.downsample1(feature_1)
    feature_3 = content_encoder.downsample2(feature_2)
    feature_4 = content_encoder.residual_blocks(feature_3)

print("原始图片：", single_image.shape)
print("输入卷积后：", feature_1.shape)
print("第一次下采样后：", feature_2.shape)
print("第二次下采样后：", feature_3.shape)
print("残差模块后：", feature_4.shape)

In [ ]:
def count_parameters(model):
    """
    计算模型中可训练参数的总数量。
    """

    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

In [ ]:
content_parameter_count = count_parameters(content_encoder)
style_parameter_count = count_parameters(style_encoder)

print(
    "内容编码器可训练参数数量：",
    f"{content_parameter_count:,}",
)

print(
    "风格编码器可训练参数数量：",
    f"{style_parameter_count:,}",
)

In [ ]:
# 取第一张内容图片的前 8 个特征通道
features_to_show = content_features[0, :8].detach().cpu()

fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for index, axis in enumerate(axes.flatten()):
    feature_map = features_to_show[index]

    axis.imshow(
        feature_map,
        cmap="viridis",
    )

    axis.set_title(f"Feature channel {index}")
    axis.axis("off")

plt.suptitle(
    "Encoded content features / 编码后的内容特征",
    fontsize=14,
)

plt.tight_layout()
plt.show()

## 11. Feature fusion and decoder / 特征融合与解码器

流程：内容图 → 内容编码器 → 空间特征；风格图 → 风格编码器 → 特征统计；AdaIN 对齐统计，再通过解码器生成 RGB 图片。

现有编码器中的 InstanceNorm 可能削弱颜色统计，因此添加原始 RGB 均值和标准差的条件支路。融合后的解码器不再使用 InstanceNorm，避免再次擦除风格统计。

以下代码也定义冻结 VGG19、训练和评估辅助函数。VGG 捕获 relu1_1、relu2_1、relu3_1、relu4_1；输入从 [-1,1] 还原到 [0,1] 再做 ImageNet 标准化，不使用分类任务的 224×224 裁剪。


In [ ]:
import csv
import json
import math
import time
def moments(x):
    return x.mean((2, 3), keepdim=True), (x.var((2, 3), unbiased=False, keepdim=True)+1e-5).sqrt()


def adain(content, style):
    cm, cs = moments(content)
    sm, ss = moments(style)
    return (content-cm)/cs * ss + sm


class Decoder(nn.Module):
    def __init__(self, channels=256):
        super().__init__()
        # Do not normalise after fusion: that would erase the injected style statistics.
        def conv(a, b):
            return nn.Sequential(nn.ReflectionPad2d(1), nn.Conv2d(a, b, 3), nn.ReLU())
        self.net = nn.Sequential(conv(channels, channels), nn.Upsample(scale_factor=2, mode='nearest'),
            conv(channels, channels//2), conv(channels//2, channels//2),
            nn.Upsample(scale_factor=2, mode='nearest'), conv(channels//2, channels//4),
            nn.ReflectionPad2d(3), nn.Conv2d(channels//4, 3, 7), nn.Tanh())

    def forward(self, x):
        return self.net(x)


class StyleTransferGenerator(nn.Module):
    def __init__(self, content_encoder=None, style_encoder=None, base_channels=64, residual_blocks=2):
        super().__init__()
        self.config = dict(base_channels=base_channels, residual_blocks=residual_blocks)
        self.content_encoder = content_encoder if content_encoder is not None else ImageEncoder(base_channels, residual_blocks)
        self.style_encoder = style_encoder if style_encoder is not None else ImageEncoder(base_channels, residual_blocks)
        channels = base_channels*4
        # Existing encoders use InstanceNorm. Add raw RGB moments so colour statistics remain accessible.
        self.style_colour = nn.Sequential(nn.Linear(6, channels), nn.ReLU(), nn.Linear(channels, channels*2))
        self.decoder = Decoder(channels)

    def forward(self, content, style, alpha=1., fusion=True):
        if not 0 <= alpha <= 1:
            raise ValueError('alpha must be between 0 and 1')
        if content.ndim != 4 or style.ndim != 4 or content.shape[0] != style.shape[0]:
            raise ValueError('Expected two BCHW image batches of equal batch size')
        if alpha == 0:
            return content
        cf = self.content_encoder(content)
        if fusion:
            sf = self.style_encoder(style)
            sm, ss = moments(style)
            scale, shift = self.style_colour(torch.cat([sm.flatten(1), ss.flatten(1)], 1)).chunk(2, 1)
            mixed = adain(cf, sf) * (1+.1*torch.tanh(scale[..., None, None])) + .1*shift[..., None, None]
        else:
            mixed = cf  # Ablation: remove both style-conditioning paths.
        output = self.decoder(mixed)
        output = F.interpolate(output, size=content.shape[-2:], mode='bilinear', align_corners=False)
        # Pixel interpolation gives exact content at alpha=0; intermediate values are a user control.
        return alpha*output + (1-alpha)*content


class VGGFeatures(nn.Module):
    """Official ImageNet VGG19, relu1_1 through relu4_1, frozen. No downloads by default."""
    def __init__(self, weights_path=None):
        super().__init__()
        layers, channels = [], 3
        for width in [64,64,'M',128,128,'M',256,256,256,256,'M',512]:
            if width == 'M':
                layers.append(nn.MaxPool2d(2))
            else:
                layers += [nn.Conv2d(channels, width, 3, padding=1), nn.ReLU(inplace=False)]
                channels = width
        self.features = nn.Sequential(*layers)
        path = Path(weights_path) if weights_path else Path(torch.hub.get_dir())/'checkpoints/vgg19-dcbb9e9d.pth'
        if not path.is_file():
            raise FileNotFoundError(f'Pretrained VGG19 required: {path}. Supply --vgg-weights. Official URL: https://download.pytorch.org/models/vgg19-dcbb9e9d.pth')
        state = torch.load(path, map_location='cpu', weights_only=True)
        selected = {k: state['features.'+k] for k in self.features.state_dict()}
        self.features.load_state_dict(selected, strict=True)
        self.requires_grad_(False)
        self.eval()
        self.register_buffer('mean', torch.tensor([.485,.456,.406]).view(1,3,1,1))
        self.register_buffer('std', torch.tensor([.229,.224,.225]).view(1,3,1,1))

    def forward(self, image):
        x = ((image+1)/2-self.mean)/self.std
        outputs = []
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i in (1,6,11,20):
                outputs.append(x)
        return outputs


def per_image_mse(a, b):
    return (a-b).square().flatten(1).mean(1)


def perceptual_metrics(extractor, output, content, style):
    of = extractor(output)  # Preserve gradients to generated pixels.
    with torch.no_grad():
        cf, sf = extractor(content), extractor(style)
    # 同时约束 relu3_1 和 relu4_1，保留更多空间结构。
    content_loss = output.new_zeros(output.shape[0])

    for index, weight in [(2, 1.0), (3, 1.0)]:
        output_mean, output_std = moments(of[index])
        content_mean, content_std = moments(cf[index])

        output_normalized = (
            of[index] - output_mean
        ) / output_std
        content_normalized = (
            cf[index] - content_mean
        ) / content_std

        content_loss += weight * per_image_mse(
        output_normalized,
        content_normalized,
    )
    style_loss = output.new_zeros(output.shape[0])
    for a, b in zip(of, sf):
        am, ast = moments(a)
        bm, bst = moments(b)
        style_loss = style_loss + per_image_mse(am,bm) + per_image_mse(ast,bst)
    tv = (output[:,:,1:]-output[:,:,:-1]).abs().flatten(1).mean(1)
    tv = tv + (output[:,:,:,1:]-output[:,:,:,:-1]).abs().flatten(1).mean(1)
    return dict(content=content_loss, style=style_loss, tv=tv)


def objective(metrics, weights):
    return sum(weights[k]*metrics[k].mean() for k in ('content','style','tv'))


def save_image(tensor, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    a = ((tensor.detach().cpu().clamp(-1,1)+1)*127.5).round().byte().permute(1,2,0).numpy()
    Image.fromarray(a).save(path)


def save_triplet(content, style, output, path):
    if style.shape[-2:] != content.shape[-2:]:
        style = F.interpolate(style[None], size=content.shape[-2:], mode='bilinear', align_corners=False)[0]
    save_image(torch.cat([content.cpu(), style.cpu(), output.cpu()], dim=2), path)


def save_checkpoint(path, model, optimizer=None, **metadata):
    payload = dict(model=model.state_dict(), config=model.config, **metadata)
    if optimizer is not None:
        payload['optimizer'] = optimizer.state_dict()
    torch.save(payload, path)


def select_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    mps = getattr(torch.backends, 'mps', None)
    if mps is not None and mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def load_generator(path, device=None):
    device = device or select_device()
    checkpoint = torch.load(path, map_location='cpu', weights_only=True)
    model = StyleTransferGenerator(**checkpoint['config']).to(device)
    model.load_state_dict(checkpoint['model'])
    model.eval()
    return model, checkpoint


@torch.no_grad()
def validate(model, loader, extractor, device, weights, max_batches=None):
    model.eval()
    totals, n = dict(content=0.,style=0.,tv=0.), 0
    for i, batch in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        c, s = batch['content'].to(device), batch['style'].to(device)
        metrics = perceptual_metrics(extractor, model(c,s), c,s)
        for key in totals:
            totals[key] += metrics[key].sum().item()
        n += len(c)
    if not n:
        raise ValueError('Empty validation loader')
    result = {k:v/n for k,v in totals.items()}
    result['total'] = sum(weights[k]*result[k] for k in totals)
    return result


def fit(model, train_loader, val_loader, extractor, device, output_dir, epochs=20, lr=1e-4,
        weights=None, max_batches=None, val_batches=None):
    weights = weights or dict(content=1.,style=10.,tv=1e-4)
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history, best = [], float('inf')
    # Separate runs should use different folders; no silent resume from a mismatched model.
    for epoch in range(1,epochs+1):
        model.train()
        sums, n = dict(content=0.,style=0.,tv=0.,total=0.), 0
        start = time.perf_counter()
        for step, batch in enumerate(train_loader):
            if max_batches is not None and step >= max_batches:
                break
            c, s = batch['content'].to(device), batch['style'].to(device)
            optimizer.zero_grad(set_to_none=True)
            generated = model(c,s)
            metrics = perceptual_metrics(extractor,generated,c,s)
            loss = objective(metrics,weights)
            if not torch.isfinite(loss):
                raise RuntimeError('Non-finite training loss')
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),5.)
            optimizer.step()
            for key in metrics:
                sums[key] += metrics[key].detach().sum().item()
            sums['total'] += loss.item()*len(c)
            n += len(c)
            # 新增：第一个 batch 和每 20 个 batch 输出进度
            if step == 0 or (step + 1) % 20 == 0:
                print(
                    f"Epoch {epoch}/{epochs}, "
                    f"batch {step + 1}/{len(train_loader)}, "
                    f"content={metrics['content'].mean().item():.4f}, "
                    f"style={metrics['style'].mean().item():.4f}, "
                    f"total={loss.item():.4f}",
                    flush=True,
                )

        if n == 0:
            raise ValueError('Empty training loader')
        val = validate(model,val_loader,extractor,device,weights,val_batches)
        row = dict(epoch=epoch,seconds=time.perf_counter()-start,
            **{'train_'+k:v/n for k,v in sums.items()}, **{'val_'+k:v for k,v in val.items()})
        history.append(row)
        meta = dict(epoch=epoch,history=history,weights=weights,lr=lr,
                    truncated_training=max_batches is not None, truncated_validation=val_batches is not None)
        save_checkpoint(out/'last.pt',model,optimizer,**meta)
        if val['total'] < best:
            best = val['total']
            save_checkpoint(out/'best.pt',model,optimizer,**meta)
        with (out/'history.csv').open('w',newline='') as f:
            writer=csv.DictWriter(f,fieldnames=list(row));writer.writeheader();writer.writerows(history)
        model.eval()
        batch = next(iter(val_loader))
        with torch.no_grad():
            c,s = batch['content'][:1].to(device),batch['style'][:1].to(device)
            sample = model(c,s)[0]
        save_triplet(c[0],s[0],sample,out/f'epoch_{epoch:03d}.png')
        print(f"Epoch {epoch}: train={row['train_total']:.4f}, val={val['total']:.4f}, {row['seconds']:.1f}s", flush=True)
    return history


@torch.no_grad()
def evaluate(model, loader, extractor, device, output_dir, max_batches=None):
    """Identical held-out pairs for identity, colour-statistics, proposed and no-fusion ablation."""
    out = Path(output_dir)
    out.mkdir(parents=True,exist_ok=True)
    model.eval()
    rows=[]
    for step,batch in enumerate(loader):
        if max_batches is not None and step >= max_batches:
            break
        c,s = batch['content'].to(device),batch['style'].to(device)
        for method in ('identity','rgb_statistics','proposed','no_fusion'):
            if device.type=='cuda': torch.cuda.synchronize()
            if device.type=='mps': torch.mps.synchronize()
            start = time.perf_counter()
            if method=='identity': y=c
            elif method=='rgb_statistics': y=adain(c,s).clamp(-1,1)
            else: y=model(c,s,fusion=method=='proposed')
            if device.type=='cuda': torch.cuda.synchronize()
            if device.type=='mps': torch.mps.synchronize()
            seconds=(time.perf_counter()-start)/len(c)
            metrics=perceptual_metrics(extractor,y,c,s)
            for i in range(len(c)):
                rows.append(dict(method=method,content_path=batch['content_path'][i],style_path=batch['style_path'][i],
                    category=Path(batch['style_path'][i]).parent.name,seconds_per_image=seconds,
                    **{key:float(value[i]) for key,value in metrics.items()}))
            if step < 4:
                save_triplet(c[0],s[0],y[0],out/f'{step:02d}_{method}.png')
    if not rows: raise ValueError('Empty evaluation loader')
    with (out/'metrics.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
    return rows


def stylize(model, content_path, style_path, output_path, size=256, alpha=1.):
    device=next(model.parameters()).device
    transform=build_transform(size)
    with Image.open(content_path) as im: c=transform(im)[None].to(device)
    with Image.open(style_path) as im: s=transform(im)[None].to(device)
    model.eval()
    with torch.no_grad(): y=model(c,s,alpha)
    save_image(y[0],output_path)
    return y



In [ ]:
# 先运行上方的数据准备和模型定义；本单元不会启动训练。
import matplotlib.pyplot as plt
PREVIEW_CHECKPOINT = RUN_DIR / 'best.pt'


if not PREVIEW_CHECKPOINT.is_file():
    print('尚无模型权重：请先运行第 13 节训练，然后重新运行本预览单元。')
else:
    preview_device = DEVICE
    if preview_device.type == 'cpu':
        torch.set_num_threads(min(4, torch.get_num_threads()))
    preview_model, preview_metadata = load_generator(PREVIEW_CHECKPOINT, preview_device)
    preview_transform = build_transform(256, training=False)
    preview_content_paths, preview_style_paths = content_test, style_test
    # 同一张内容图搭配不同风格，便于直接比较。
    preview_content_path = preview_content_paths[0]
    preview_style_indices = [0, len(preview_style_paths)//2, len(preview_style_paths)-1]
    preview_epoch = preview_metadata.get('epoch', '?')
    print(f'Checkpoint: {PREVIEW_CHECKPOINT}')
    print(f'Saved checkpoint epoch: {preview_epoch}; device: {preview_device}')
    if preview_metadata.get('truncated_training'):
        print('此权重仅做过少量 batch 的流程测试，不代表最终风格迁移质量。')
    else:
        print('下面展示当前已保存权重的真实输出；训练早期可能模糊，不能视为收敛结果。')

    from PIL import Image
    with Image.open(preview_content_path) as image:
        preview_content = preview_transform(image)[None].to(preview_device)
    preview_figure, preview_axes = plt.subplots(3, 3, figsize=(10, 10), squeeze=False)
    with torch.inference_mode():
        for row, style_index in enumerate(preview_style_indices):
            style_path = preview_style_paths[style_index]
            with Image.open(style_path) as image:
                preview_style = preview_transform(image)[None].to(preview_device)
            preview_output = preview_model(preview_content, preview_style, alpha=1.0)
            for column, (tensor, title) in enumerate([
                (preview_content[0], 'Content'),
                (preview_style[0], 'Style: '+style_path.parent.name.replace('_', ' ')),
                (preview_output[0], f'Generated (checkpoint epoch {preview_epoch})'),
            ]):
                ax = preview_axes[row, column]
                ax.imshow(((tensor.detach().cpu()+1)/2).clamp(0,1).permute(1,2,0))
                ax.set_title(title, fontsize=10)
                ax.axis('off')
    preview_figure.suptitle('Image Style Transfer: Content / Reference Style / Generated', fontsize=13)
    preview_figure.tight_layout(rect=(0,0,1,.96))
    display(preview_figure)
    plt.close(preview_figure)
    # 释放预览模型显存，避免影响后续训练。
    del preview_model


In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

# 预览可以在训练前安全执行；只有当前实验已保存权重时才加载。
if 'RUN_DIR' not in globals():
    print('请先运行环境配置单元，定义 RUN_NAME 和 RUN_DIR。')
elif not (RUN_DIR / 'best.pt').is_file():
    print(f'尚无当前实验的训练权重：{RUN_DIR / "best.pt"}')
    print('请继续运行第 12、13 节完成初始化和训练，然后重新运行本预览单元。')
else:
    # 加载本次实验保存的最佳模型
    preview_model, metadata = load_generator(
        RUN_DIR / 'best.pt', DEVICE
    )
    transform = build_transform(256, training=False)

    # 改 SEED 可以换一批图片；固定它便于比较不同模型
    SEED_PREVIEW = 3
    rng = random.Random(SEED_PREVIEW)

    number = min(4, len(content_test), len(style_test))
    if number == 0:
        raise ValueError("测试图片为空，请先运行数据划分单元。")

    content_paths = rng.sample(list(content_test), number)
    style_paths = rng.sample(list(style_test), number)

    fig, axes = plt.subplots(
        number, 3, figsize=(12, 4 * number), squeeze=False
    )

    preview_model.eval()
    with torch.inference_mode():
        for row, (content_path, style_path) in enumerate(
            zip(content_paths, style_paths)
        ):
            with Image.open(content_path) as image:
                content = transform(image.convert("RGB"))[None].to(DEVICE)

            with Image.open(style_path) as image:
                style = transform(image.convert("RGB"))[None].to(DEVICE)

            output = preview_model(content, style, alpha=1.0)

            for col, (tensor, title) in enumerate([
                (content[0], "Content"),
                (style[0], "Style"),
                (output[0], "Generated"),
            ]):
                image = ((tensor.detach().cpu() + 1) / 2).clamp(0, 1)
                axes[row, col].imshow(image.permute(1, 2, 0))
                axes[row, col].set_title(title)
                axes[row, col].axis("off")

    plt.tight_layout()
    plt.show()

    # 释放额外加载的模型
    del preview_model


## 12. Generator and perceptual objective / 生成器与感知损失

损失：标准化 relu4_1 内容特征 MSE + 10×多层风格均值/标准差 MSE + 0.0001×总变差损失。VGG 权重冻结，但输出图片经过 VGG 的计算图保留，用于向生成器反向传播。

VGG 损失是代理指标，不等同于人类审美评价；没有成对的风格化真值图，不用 PSNR/SSIM 作为主要质量结论。


In [ ]:
import time

# 在模型初始化前设置，避免 CPU 线程过多造成严重变慢。
if DEVICE.type == 'cpu':
    torch.set_num_threads(min(4, torch.get_num_threads()))

_setup_start = time.perf_counter()
def setup_progress(message):
    print(f'[{time.perf_counter() - _setup_start:.1f}s] {message}', flush=True)

setup_progress(f'1/5 创建生成器；设备：{DEVICE}')
generator = StyleTransferGenerator(
    content_encoder=content_encoder,
    style_encoder=style_encoder,
    base_channels=64,
    residual_blocks=2,
).to(DEVICE)
setup_progress(f'生成器创建完成，参数量：{count_parameters(generator):,}')

# 预训练权重统一放在 model/；已有 PyTorch 缓存则复制，否则下载官方文件。
VGG_WEIGHTS = PRETRAINED_ROOT / 'vgg19-dcbb9e9d.pth'
_cached_vgg = Path(torch.hub.get_dir()) / 'checkpoints' / VGG_WEIGHTS.name
if not VGG_WEIGHTS.is_file() and _cached_vgg.is_file():
    import shutil
    VGG_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(_cached_vgg, VGG_WEIGHTS)
_vgg_path = Path(VGG_WEIGHTS)
setup_progress(f'2/5 准备 VGG19 权重：{_vgg_path}')
if not _vgg_path.is_file():
    _vgg_path.parent.mkdir(parents=True, exist_ok=True)
    setup_progress('首次下载官方 VGG19 权重（约 548 MiB），请等待下载进度完成')
    try:
        torch.hub.download_url_to_file(
            'https://download.pytorch.org/models/vgg19-dcbb9e9d.pth',
            str(_vgg_path), hash_prefix='dcbb9e9d', progress=True,
        )
    except Exception as error:
        raise RuntimeError(
            f'VGG19 权重下载失败。可重新运行重试，或手动下载到 {_vgg_path}。原始错误：{error}'
        ) from error
setup_progress(f'权重文件大小：{_vgg_path.stat().st_size / 1024**2:.1f} MiB')
perceptual_net = VGGFeatures(VGG_WEIGHTS).to(DEVICE)
LOSS_WEIGHTS = dict(content=6., style=10., tv=1e-4)
setup_progress('VGG19 加载完成')

setup_progress('3/5 读取一个验证样本（直接读取 Dataset，避免等待整批图片）')
sample = val_loader.dataset[0]
c = sample['content'].unsqueeze(0).to(DEVICE)
s = sample['style'].unsqueeze(0).to(DEVICE)
setup_progress(f'样本读取完成：content={tuple(c.shape)}, style={tuple(s.shape)}')

setup_progress('4/5 生成器前向计算')
_previous_training = generator.training
generator.eval()
try:
    with torch.no_grad():
        generated = generator(c, s)
        # 转到 CPU 时等待设备完成计算，便于确认这一阶段已结束。
        _generated_cpu = generated.detach().cpu()
        assert generated.shape == c.shape
        setup_progress(f'前向计算完成：{tuple(generated.shape)}')
        setup_progress('5/5 检查 alpha=0 是否保留原图')
        assert torch.equal(generator(c, s, alpha=0), c)
finally:
    generator.train(_previous_training)
setup_progress('初始化及前向检查全部完成，可以继续训练')


## 13. Training and checkpoint selection / 训练与验证

训练集更新参数；每轮使用完整验证集选择 `best.pt`，测试集不参与选模型。`last.pt` 也保存模型、优化器和配置，可供后续扩展恢复训练使用；当前训练函数每次从新优化器开始，不会自动恢复。

请为不同实验选择不同 RUN_NAME。先用 EPOCHS=1、MAX_BATCHES=2、VAL_BATCHES=1 检查设备，再把两个 batch 限制恢复为 None 做正式训练。20 个 epoch 是初始预算，并不保证收敛，应根据验证曲线决定是否增加。


In [ ]:
# RUN_NAME 和 RUN_DIR 已在环境配置单元中统一定义。
EPOCHS = 50
MAX_BATCHES = None
VAL_BATCHES = None
LEARNING_RATE = 1e-4
if DEVICE.type == 'cpu':
    torch.set_num_threads(min(4, torch.get_num_threads()))
seed_everything(SEED)
history = fit(generator, train_loader, val_loader, perceptual_net, DEVICE,
              RUN_DIR, epochs=EPOCHS, lr=LEARNING_RATE, weights=LOSS_WEIGHTS,
              max_batches=MAX_BATCHES, val_batches=VAL_BATCHES)


In [ ]:
history_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key in zip(axes, ['total', 'content', 'style']):
    ax.plot(history_df.epoch, history_df['train_'+key], label='Train')
    ax.plot(history_df.epoch, history_df['val_'+key], label='Validation')
    ax.set(xlabel='Epoch', ylabel=key, title=key.capitalize()+' loss')
    ax.legend()
plt.tight_layout()
plt.show()


## 14. Held-out comparison and ablation / 留出测试、基线与消融

所有方法使用完全相同的固定内容—风格配对。覆盖全部 213 张测试风格图，部分内容图会重复。分别报告逐对均值、逐类别结果，防止主流类别掩盖少数类别。

- identity：不修改内容图，是内容误差的下界参考。
- rgb_statistics：只对齐 RGB 均值/标准差的廉价基线，不代表神经 AdaIN 方法。
- proposed：完整训练后的生成器。
- no_fusion：同一模型推理时移除风格支路，用于诊断条件依赖。它不是独立重新训练的消融模型；如需严谨比较，应另外训练一个无融合模型。

速度不包含文件加载、VGG 评分；此处包括第一次调用的冷启动开销，只作粗略参考。正式计时应预热、多次重复并报告硬件。


In [ ]:
best_generator, checkpoint_info = load_generator(RUN_DIR / 'best.pt', DEVICE)
if checkpoint_info.get('truncated_training') or checkpoint_info.get('truncated_validation'):
    print('WARNING: truncated smoke run; these are not final project results.')
rows = evaluate(best_generator, test_loader, perceptual_net, DEVICE, RUN_DIR / 'test')
results_df = pd.DataFrame(rows)
display(results_df.groupby('method')[['content', 'style', 'tv', 'seconds_per_image']].mean())
category_results = results_df.groupby(['method', 'category'])[['content', 'style']].mean()
display(category_results)
category_results.to_csv(RUN_DIR / 'test' / 'category_metrics.csv')
print('Macro style error (equal weight per represented category):')
display(category_results.groupby('method')[['content', 'style']].mean())


In [ ]:
# Same content with multiple held-out style references.
fixed_content = test_dataset[0]['content'][None].to(DEVICE)
indices = np.linspace(0, len(test_dataset)-1, min(4,len(test_dataset)), dtype=int)
fig, axes = plt.subplots(len(indices), 3, figsize=(9, 3*len(indices)), squeeze=False)
best_generator.eval()
with torch.no_grad():
    for row, index in enumerate(indices):
        st = test_dataset[int(index)]['style'][None].to(DEVICE)
        pred = best_generator(fixed_content, st)
        for ax, image, title in zip(axes[row], [fixed_content[0],st[0],pred[0]], ['Content','Style','Generated']):
            ax.imshow(((image.detach().cpu()+1)/2).clamp(0,1).permute(1,2,0))
            ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()


## 15. Inference and style strength / 推理与风格强度

alpha=0 精确返回内容图，alpha=1 返回完整生成结果，中间值为两者的像素插值。这里采用固定中心裁剪，与评估预处理一致，可能丢失边缘内容；生成器本身支持非正方形和奇数尺寸，后续可扩展保留长宽比的输入预处理。


In [ ]:
CONTENT_PATH = content_test[0]
STYLE_PATH = style_test[0]
OUTPUT_PATH = RUN_DIR / 'my_stylized_image.png'
_ = stylize(best_generator, CONTENT_PATH, STYLE_PATH, OUTPUT_PATH, size=256, alpha=1.)
display(Image.open(OUTPUT_PATH))


## 16. Discussion and experiment checklist / 讨论与实验记录

完成正式训练后，根据实际生成图与指标填写分析，不预设本模型一定胜过基线。

1. 报告最低验证损失对应 epoch、测试内容误差/风格误差、逐类别及宏平均结果，展示同内容多风格和同风格多内容。
2. 比较 identity 与 RGB 统计基线；若完整模型主要改变颜色、无法生成笔触，应如实报告，考虑更长训练或更大数据量。
3. 测试集存在训练/验证未充分覆盖的稀有风格。保留原分割便于对照，不将这些类别的表现混同于充分训练类别。
4. 两个独立可训练编码器并不保证语义解耦。RGB 统计支路可能主导条件控制，建议分别重新训练无 RGB 支路、无特征融合的模型。
5. 本次 smoke run 只有 2 个训练 batch，不能据此判断收敛、风格质量或达到评分要求。
6. 局限：小数据集、128×128 训练、裁剪损失、VGG 指标偏差、风格类别不平衡；未来可比较预训练编码器和 Transformer 方案，但本实现不声称达到 StyTr² 或 CycleGAN 性能。
7. 正式报告、展示和个人贡献说明应基于小组实际实验填写；代码完成不等于整套课程提交已完成。
